# Predição do modelo de classificação de coral-sol, terceira e quarta versão, nas provas de foto (1, 2 e 3)

Autor:  Viviane da Rosa Sommer

Atualizado: 19/03/2025

## Objetivo:
Notebook destinado à predição das imagens das provas de foto utilizando o modelo V3 e V4 de classificação de coral-sol, com o objetivo de analisar seu desempenho.

## Importações Necessárias

In [ ]:
import numpy as np
import cv2
import glob
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## Declaração de Constantes e Modelos

In [ ]:
model_v1 = YOLO('Pesos/best_weights_v3.pt') 
model_v2 = YOLO('runs/classify/train6/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Reads an image, converts it to RGB format, and performs a central crop 
    keeping the original resolution.

    Args:
        filename (str): Path to the image file.

    Returns:
        np.ndarray: Cropped image in RGB format. Returns None if an error occurs.
    """
    try:
        image = cv2.imread(filename)
        if image is None:
            print(f"Error loading image: {filename}")
            return None
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width, _ = image.shape
        mid = height // 2
        top = max(0, mid - int(0.34 * height))
        bottom = min(height, mid + int(0.34 * height))
        cropped_image = image[top:bottom, :]
        return cropped_image.astype(np.uint8)
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None

def plot_results(image: np.ndarray, true_label: str, predicted_label_v1: str, 
                 predicted_label_v2: str, v1_score: float, v2_score: float) -> None:
    """
    Plots the classification results for the given image.

    Args:
        image (np.ndarray): Image to be displayed.
        true_label (str): True label of the image.
        predicted_label_v1 (str): Predicted label by Model V1.
        predicted_label_v2 (str): Predicted label by Model V2.
        v1_score (float): Confidence score for Model V1's prediction.
        v2_score (float): Confidence score for Model V2's prediction.

    Returns:
        None
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(image)
    axes[0].set_title(f"True label: {true_label}", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(image)
    axes[1].set_title(f"Model V1 Prediction: {predicted_label_v1} ({v1_score:.2f})", fontsize=14)
    axes[1].axis('off')
    
    axes[2].imshow(image)
    axes[2].set_title(f"Model V2 Prediction: {predicted_label_v2} ({v2_score:.2f})", fontsize=14)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

def evaluate_classification(true_labels: list[str], predicted_labels_v1: list[str], predicted_labels_v2: list[str]) -> None:
    """
    Evaluates classification performance by generating confusion matrices and classification reports.

    Args:
        true_labels (list[str]): The true labels.
        predicted_labels_v1 (list[str]): Predictions from Model V1.
        predicted_labels_v2 (list[str]): Predictions from Model V2.

    Returns:
        None
    """
    true_labels_binary = [1 if label == 'Positive' else 0 for label in true_labels]
    predicted_labels_binary_v1 = [1 if label == 'Positive' else 0 for label in predicted_labels_v1]
    predicted_labels_binary_v2 = [1 if label == 'Positive' else 0 for label in predicted_labels_v2]

    conf_matrix_v1 = confusion_matrix(true_labels_binary, predicted_labels_binary_v1)
    disp_v1 = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_v1, display_labels=['Negative', 'Positive'])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    disp_v1.plot(cmap=plt.cm.Blues, ax=ax1)
    ax1.set_title("Confusion Matrix - V1 Model")

    conf_matrix_v2 = confusion_matrix(true_labels_binary, predicted_labels_binary_v2)
    disp_v2 = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_v2, display_labels=['Negative', 'Positive'])
    disp_v2.plot(cmap=plt.cm.Blues, ax=ax2)
    ax2.set_title("Confusion Matrix - V2 Model")

    plt.tight_layout()
    plt.show()

    print("Classification Report - V1 Model:")
    print(classification_report(true_labels_binary, predicted_labels_binary_v1, target_names=['Negative', 'Positive']))
    print("\nClassification Report - V2 Model:")
    print(classification_report(true_labels_binary, predicted_labels_binary_v2, target_names=['Negative', 'Positive']))

## Processamento das imagens - OS 6000696049

In [ ]:
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto _6000696049"
true_labels = []
predicted_labels_v1 = []
predicted_labels_v2 = []

for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**", recursive=True):
    try:
        if not os.path.isfile(filename):
            continue

        image = read_image(filename)
        if image is None:
            continue

        results_v1 = model_v1.predict(filename, verbose=False)
        class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()
        predicted_label_index_v1 = np.argmax(class_probabilities_v1)
        v1_score = class_probabilities_v1[predicted_label_index_v1]
        predicted_label_v1 = 'Positive' if predicted_label_index_v1 == 1 else 'Negative'

        results_v2 = model_v2.predict(filename, verbose=False)
        class_probabilities_v2 = results_v2[0].probs.data.cpu().numpy()
        predicted_label_index_v2 = np.argmax(class_probabilities_v2)
        v2_score = class_probabilities_v2[predicted_label_index_v2]
        predicted_label_v2 = 'Positive' if predicted_label_index_v2 == 1 else 'Negative'

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        true_labels.append(true_label)
        predicted_labels_v1.append(predicted_label_v1)
        predicted_labels_v2.append(predicted_label_v2)

        plot_results(
            image=image,
            true_label=true_label,
            predicted_label_v1=predicted_label_v1,
            predicted_label_v2=predicted_label_v2,
            v1_score=v1_score,
            v2_score=v2_score
        )

    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Matriz de confusão - OS 6000696049

In [ ]:
evaluate_classification(true_labels, predicted_labels_v1, predicted_labels_v2) 

## Processamento das imagens - OS 6000504195

In [ ]:
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto_6000504195/"
true_labels = []
predicted_labels_v1 = []
predicted_labels_v2 = []

for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**", recursive=True):
    try:
        if not os.path.isfile(filename):
            continue

        image = read_image(filename)
        if image is None:
            continue

        results_v1 = model_v1.predict(filename, verbose=False)
        class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()
        predicted_label_index_v1 = np.argmax(class_probabilities_v1)
        v1_score = class_probabilities_v1[predicted_label_index_v1]
        predicted_label_v1 = 'Positive' if predicted_label_index_v1 == 1 else 'Negative'

        results_v2 = model_v2.predict(source=filename, save=False, verbose=False)
        class_probabilities_v2 = results_v2[0].probs.data.cpu().numpy()
        predicted_label_index = np.argmax(class_probabilities_v2) 
        v2_score = class_probabilities_v2[predicted_label_index] 
        predicted_label_v2 = 'Positive' if predicted_label_index == 1 else 'Negative'

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        true_labels.append(true_label)
        predicted_labels_v1.append(predicted_label_v1)
        predicted_labels_v2.append(predicted_label_v2)

        plot_results(
            image=image,
            true_label=true_label,
            predicted_label_v1=predicted_label_v1,
            predicted_label_v2=predicted_label_v2,
            v1_score=v1_score,
            v2_score=v2_score
        )

    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Matriz de confusão - OS 6000504195

In [ ]:
evaluate_classification(true_labels, predicted_labels_v1, predicted_labels_v2) 

## Processamento das imagens - OS 6000701944

In [ ]:
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de Foto_6000701944/"
true_labels = []
predicted_labels_v1 = []
predicted_labels_v2 = []

for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**", recursive=True):
    try:
        if not os.path.isfile(filename):
            continue

        image = read_image(filename)
        if image is None:
            continue

        results_v1 = model_v1.predict(filename, verbose=False)
        class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()
        predicted_label_index_v1 = np.argmax(class_probabilities_v1)
        v1_score = class_probabilities_v1[predicted_label_index_v1]
        predicted_label_v1 = 'Positive' if predicted_label_index_v1 == 1 else 'Negative'

        results_v2 = model_v2.predict(source=filename, save=False, verbose=False)
        class_probabilities_v2 = results_v2[0].probs.data.cpu().numpy()
        predicted_label_index = np.argmax(class_probabilities_v2) 
        v2_score = class_probabilities_v2[predicted_label_index] 
        predicted_label_v2 = 'Positive' if predicted_label_index == 1 else 'Negative'

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        true_labels.append(true_label)
        predicted_labels_v1.append(predicted_label_v1)
        predicted_labels_v2.append(predicted_label_v2)

        plot_results(
            image=image,
            true_label=true_label,
            predicted_label_v1=predicted_label_v1,
            predicted_label_v2=predicted_label_v2,
            v1_score=v1_score,
            v2_score=v2_score
        )

    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Matriz de confusão - OS 6000701944

In [ ]:
evaluate_classification(true_labels, predicted_labels_v1, predicted_labels_v2) 

In [ ]:
!jupyter nbconvert --to html prediction_prova_foto-comparacao.ipynb